# NGLab Tutorial #9: Backtesting Framework

Learn to evaluate trained agents on historical data with realistic market conditions.

## Learning Objectives

1. Load historical price data
2. Configure slippage and transaction costs
3. Run backtests on trained agents
4. Generate performance reports (Sharpe, Drawdown, Win Rate)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
print("Libraries loaded successfully!")

## 1. Generate Historical Data

Create 2 years of realistic BTC price data:

In [ ]:
# Generate synthetic price history
n_days = 730  # 2 years
dates = pd.date_range('2022-01-01', periods=n_days, freq='D')

# Geometric Brownian Motion
drift = 0.0003  # Daily drift
volatility = 0.025  # Daily volatility
returns = np.random.normal(drift, volatility, n_days)
prices = 30000 * np.exp(np.cumsum(returns))

# Create DataFrame
df = pd.DataFrame({
    'date': dates,
    'price': prices,
    'volume': np.random.uniform(100, 1000, n_days)
})

print(f"Generated {len(df)} days of historical data")
print(f"Price range: ${df['price'].min():,.2f} - ${df['price'].max():,.2f}")
print(f"Total return: {(df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100:.2f}%")

In [ ]:
# Visualize data
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.plot(df['date'], df['price'], linewidth=1.5, color='steelblue')
ax1.set_title('BTC Historical Prices (2022-2024)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, alpha=0.3)

df['returns'] = df['price'].pct_change()
ax2.hist(df['returns'].dropna(), bins=50, color='coral', alpha=0.7, edgecolor='black')
ax2.set_title('Returns Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Daily Return')
ax2.set_ylabel('Frequency')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Backtesting Engine

In [ ]:
class Backtester:
    def __init__(self, data, initial_capital=100000, transaction_cost=0.001, slippage_bps=5):
        self.data = data
        self.initial_capital = initial_capital
        self.transaction_cost = transaction_cost  # 0.1%
        self.slippage_bps = slippage_bps  # 5 basis points
    
    def run(self, agent_func, lookback=60):
        """Run backtest with given agent.
        
        Args:
            agent_func: Function(observation) -> action (0/1/2)
            lookback: Observation window size
        """
        cash = self.initial_capital
        position = 0.0
        
        trades = []
        portfolio_values = []
        
        for i in range(lookback, len(self.data)):
            # Get observation
            obs = self.data['price'].iloc[i-lookback:i].values
            price = self.data['price'].iloc[i]
            
            # Agent decision
            action = agent_func(obs)
            
            # Execute trade
            if action == 1 and cash >= price * 0.1:  # Buy 0.1 BTC
                # Apply slippage (pay more)
                exec_price = price * (1 + self.slippage_bps / 10000)
                cost = exec_price * 0.1
                fee = cost * self.transaction_cost
                
                if cash >= cost + fee:
                    position += 0.1
                    cash -= (cost + fee)
                    trades.append({'date': self.data['date'].iloc[i], 'action': 'BUY', 'price': exec_price})
            
            elif action == 2 and position >= 0.1:  # Sell 0.1 BTC
                # Apply slippage (get less)
                exec_price = price * (1 - self.slippage_bps / 10000)
                proceeds = exec_price * 0.1
                fee = proceeds * self.transaction_cost
                
                position -= 0.1
                cash += (proceeds - fee)
                trades.append({'date': self.data['date'].iloc[i], 'action': 'SELL', 'price': exec_price})
            
            # Record portfolio value
            portfolio_value = cash + position * price
            portfolio_values.append({
                'date': self.data['date'].iloc[i],
                'value': portfolio_value,
                'cash': cash,
                'position': position,
                'price': price
            })
        
        return pd.DataFrame(portfolio_values), pd.DataFrame(trades)

print("Backtester class defined!")

## 3. Define Trading Strategies

In [ ]:
# Simple Moving Average Crossover
def sma_crossover_agent(obs):
    if len(obs) < 20:
        return 0
    
    sma_short = np.mean(obs[-5:])
    sma_long = np.mean(obs[-20:])
    
    if sma_short > sma_long * 1.01:
        return 1  # Buy
    elif sma_short < sma_long * 0.99:
        return 2  # Sell
    return 0

# Momentum strategy
def momentum_agent(obs):
    if len(obs) < 10:
        return 0
    
    momentum = (obs[-1] - obs[-10]) / obs[-10]
    
    if momentum > 0.02:  # 2% gain
        return 1
    elif momentum < -0.02:  # 2% loss
        return 2
    return 0

# Buy and Hold
def buy_hold_agent(obs):
    return 1  # Always buy (if possible)

print("Agent strategies defined!")

## 4. Run Backtests

In [ ]:
backtester = Backtester(df)

# Run for all strategies
results_sma, trades_sma = backtester.run(sma_crossover_agent)
results_mom, trades_mom = backtester.run(momentum_agent)
results_bh, trades_bh = backtester.run(buy_hold_agent)

print(f"\n=== Backtest Complete ===")
print(f"SMA Crossover: {len(trades_sma)} trades")
print(f"Momentum: {len(trades_mom)} trades")
print(f"Buy & Hold: {len(trades_bh)} trades")

## 5. Performance Metrics

In [ ]:
def calculate_metrics(results, initial_capital=100000):
    """Calculate performance metrics."""
    final_value = results['value'].iloc[-1]
    total_return = (final_value / initial_capital - 1) * 100
    
    # Daily returns
    daily_returns = results['value'].pct_change().dropna()
    
    # Sharpe ratio (annualized)
    sharpe = np.sqrt(252) * (daily_returns.mean() / daily_returns.std()) if daily_returns.std() > 0 else 0
    
    # Max drawdown
    cummax = results['value'].cummax()
    drawdown = (results['value'] - cummax) / cummax
    max_drawdown = drawdown.min() * 100
    
    return {
        'total_return': total_return,
        'sharpe_ratio': sharpe,
        'max_drawdown': max_drawdown,
        'final_value': final_value
    }

# Calculate for all strategies
metrics_sma = calculate_metrics(results_sma)
metrics_mom = calculate_metrics(results_mom)
metrics_bh = calculate_metrics(results_bh)

print("\n=== Performance Metrics ===")
print(f"{'Strategy':<20} {'Return':>10} {'Sharpe':>8} {'Max DD':>10} {'Final Value':>15}")
print("-" * 70)
print(f"{'SMA Crossover':<20} {metrics_sma['total_return']:>9.2f}% {metrics_sma['sharpe_ratio']:>8.2f} {metrics_sma['max_drawdown']:>9.2f}% ${metrics_sma['final_value']:>13,.2f}")
print(f"{'Momentum':<20} {metrics_mom['total_return']:>9.2f}% {metrics_mom['sharpe_ratio']:>8.2f} {metrics_mom['max_drawdown']:>9.2f}% ${metrics_mom['final_value']:>13,.2f}")
print(f"{'Buy & Hold':<20} {metrics_bh['total_return']:>9.2f}% {metrics_bh['sharpe_ratio']:>8.2f} {metrics_bh['max_drawdown']:>9.2f}% ${metrics_bh['final_value']:>13,.2f}")

In [ ]:
# Visualize results
plt.figure(figsize=(14, 6))
plt.plot(results_sma['date'], results_sma['value'], label='SMA Crossover', linewidth=2)
plt.plot(results_mom['date'], results_mom['value'], label='Momentum', linewidth=2)
plt.plot(results_bh['date'], results_bh['value'], label='Buy & Hold', linewidth=2)
plt.axhline(y=100000, color='red', linestyle='--', alpha=0.5, label='Initial Capital')

plt.title('Strategy Comparison: Portfolio Value Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

In this notebook, you learned:

✅ Backtesting framework with realistic costs  
✅ Slippage and transaction cost modeling  
✅ Performance metrics (Sharpe, Drawdown)  
✅ Strategy comparison and visualization  

## Next Steps

Complete the series with **Notebook #10**: Advanced Topics!

---